# Notebook 03 — Random Forest Regressor
**Author:** Ahmed Deraz (24046227)
Trains and evaluates a Random Forest model on the Health Risk Score prediction task.

In [ ]:
import sys, os, json
sys.path.append("../src")
from preprocessing import load_and_preprocess
from rf_model import train_rf, evaluate, cross_validate_rf

import numpy as np
import matplotlib.pyplot as plt

os.makedirs("../report/figures", exist_ok=True)

X_train, X_test, y_train, y_test, feature_names = load_and_preprocess(filepath="../data/urban_air_quality.csv")
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Features: {len(feature_names)}")
print(f"Baseline to beat → MAE=0.1079  RMSE=0.1326")

In [ ]:
# Cell 2 — Tune n_estimators
print("5-fold CV: tuning n_estimators...
")
results = []
for n_est in [50, 100, 200, 300, 500]:
    mean_mae, std_mae = cross_validate_rf(X_train, y_train, n_estimators=n_est, cv=5)
    results.append((n_est, mean_mae, std_mae))
    print(f"  n_estimators={n_est:>3} | CV MAE = {mean_mae:.4f} ± {std_mae:.4f}")

best_n = min(results, key=lambda x: x[1])[0]
print(f"
✅ Best n_estimators: {best_n}")

In [ ]:
# Cell 3 — Tune max_depth
print(f"
Tuning max_depth (n_estimators={best_n})...
")
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
depth_results = []
for depth in [None, 5, 10, 15, 20]:
    m = RandomForestRegressor(n_estimators=best_n, max_depth=depth, random_state=42, n_jobs=-1)
    scores = cross_val_score(m, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
    mean_mae = -scores.mean()
    depth_results.append((depth, mean_mae))
    print(f"  max_depth={str(depth):<6} | CV MAE = {mean_mae:.4f}")

best_depth = min(depth_results, key=lambda x: x[1])[0]
print(f"
✅ Best max_depth: {best_depth}")

In [ ]:
# Cell 4 — Train Final Model
rf_model = train_rf(X_train, y_train, n_estimators=best_n, max_depth=best_depth)
metrics = evaluate(rf_model, X_test, y_test)
y_pred_rf = metrics["y_pred"]

print("=" * 55)
print("  RANDOM FOREST — FINAL TEST RESULTS")
print("=" * 55)
print(f"  MAE  : {metrics['mae']:.4f}  (baseline: 0.1079)")
print(f"  RMSE : {metrics['rmse']:.4f}  (baseline: 0.1326)")
improvement = (0.1079 - metrics["mae"]) / 0.1079 * 100
print(f"  MAE improvement over baseline: {improvement:.1f}%")
if metrics["mae"] < 0.1079:
    print("  ✅ Beat the baseline!")
else:
    print("  ⚠️  Linear Regression outperforms RF on this dataset")
print("=" * 55)

In [ ]:
# Cell 5 — Feature Importance Chart
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(11, 5))
plt.bar(range(len(indices)), importances[indices], color="steelblue", edgecolor="black")
plt.xticks(range(len(indices)), [feature_names[i] for i in indices], rotation=45, ha="right", fontsize=9)
plt.title(f"Random Forest — Top 15 Feature Importances
(n_estimators={best_n}, max_depth={best_depth})", fontsize=12)
plt.ylabel("Importance Score")
plt.tight_layout()
plt.savefig("../report/figures/rf_feature_importance.png", dpi=150)
plt.show()
print("Saved → report/figures/rf_feature_importance.png")
print("
Top 5 features:")
for i in indices[:5]:
    print(f"  {feature_names[i]:<25} {importances[i]:.4f}")

In [ ]:
# Cell 6 — Predicted vs Actual Scatter
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_rf, alpha=0.5, s=20, color="steelblue", edgecolors="white", linewidths=0.3)
lims = [min(y_test.min(), y_pred_rf.min()) - 0.1, max(y_test.max(), y_pred_rf.max()) + 0.1]
plt.plot(lims, lims, "r--", linewidth=2, label="Perfect prediction")
plt.xlabel("Actual Health Risk Score")
plt.ylabel("Predicted Health Risk Score")
plt.title(f"Random Forest: Predicted vs Actual
MAE={metrics['mae']:.4f}   RMSE={metrics['rmse']:.4f}")
plt.legend()
plt.tight_layout()
plt.savefig("../report/figures/rf_scatter.png", dpi=150)
plt.show()
print("Saved → report/figures/rf_scatter.png")

In [ ]:
# Cell 7 — Time-Series Plot
plt.figure(figsize=(14, 4))
plt.plot(y_test,    label="Actual",       color="black",     linewidth=1.5)
plt.plot(y_pred_rf, label="RF Predicted", color="steelblue", linewidth=1.2, alpha=0.85)
plt.title("Random Forest — Test Set Predictions (200 records)")
plt.xlabel("Test Record Index")
plt.ylabel("Health Risk Score")
plt.legend()
plt.tight_layout()
plt.savefig("../report/figures/rf_timeseries.png", dpi=150)
plt.show()
print("Saved → report/figures/rf_timeseries.png")

In [ ]:
# Cell 8 — Save Results JSON
rf_results = {
    "model": "Random Forest",
    "best_n_estimators": int(best_n),
    "best_max_depth": str(best_depth),
    "mae":  metrics["mae"],
    "rmse": metrics["rmse"],
    "top_5_features": [feature_names[i] for i in indices[:5]]
}

with open("../report/rf_results.json", "w") as f:
    json.dump(rf_results, f, indent=2)

print("✅ Saved → report/rf_results.json")
print(rf_results)
print("
→ Message John: rf_results.json is ready. He can now run notebook 05.")